In [84]:
import torchfrom torch import nnimport torchvisionimport torchvision.transforms as transformsfrom torch.utils.data import DataLoaderfrom torch_fidelity import calculate_metricsimport warningsimport osfrom torchvision.utils import save_imageimport matplotlib.pylab as pltfrom tqdm.auto import tqdmfrom timeit import default_timer as timerimport torch.nn.functional as Fimport numpy as npimport pandas as pdfrom sklearn.manifold import TSNEimport seaborn as snswarnings.filterwarnings('ignore')

## Device

In [85]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Timer

In [86]:
def print_train_time(start: float, end : float, device: torch.device = None):    total_time = end - start    print(f"Train time on {device}: {total_time/60:.3f} minutes\n\n")    return total_time

## Data

In [87]:
transform = transforms.Compose([    transforms.ToTensor(),    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])# Download and load the training datatrainset = torchvision.datasets.CIFAR10(root='./data', train=True,                                        download=True, transform=transform)# Download and load the testing datatestset = torchvision.datasets.CIFAR10(root='./data', train=False,                                       download=True, transform=transform)

In [88]:
# DataLoadersbatch_size = 128 # batch size of the original paperdataloader = DataLoader(trainset, batch_size=batch_size, shuffle=True)

In [ ]:
class Generator(nn.Module):    def __init__(self, z_dim, channels_img, features_g, use_batch_norm=True, activation_type="relu"):        super(Generator, self).__init__()        if activation_type == "relu":            act_fn = nn.ReLU(True)        elif activation_type == "leaky_relu":            act_fn = nn.LeakyReLU(0.2, inplace=True)        elif activation_type == "tanh":            act_fn = nn.Tanh()        else:            raise ValueError(f"Unsupported activation type: {activation_type}")        self.gen = nn.Sequential(            # Layer 1            nn.ConvTranspose2d(z_dim, features_g * 4, kernel_size=4, stride=1, padding=0, bias=not use_batch_norm),            *([nn.BatchNorm2d(features_g * 4)] if use_batch_norm else []),            act_fn,            # Layer 2            nn.ConvTranspose2d(features_g * 4, features_g * 2, kernel_size=4, stride=2, padding=1, bias=not use_batch_norm),            *([nn.BatchNorm2d(features_g * 2)] if use_batch_norm else []),            act_fn,            # Layer 3            nn.ConvTranspose2d(features_g * 2, features_g, kernel_size=4, stride=2, padding=1, bias=not use_batch_norm),            *([nn.BatchNorm2d(features_g)] if use_batch_norm else []),            act_fn,            # Layer 4 (Output)            nn.ConvTranspose2d(features_g, channels_img, kernel_size=4, stride=2, padding=1, bias=True), # No BN on output            nn.Tanh(),        )    def forward(self, x):        return self.gen(x)

In [ ]:
class Discriminator(nn.Module):    def __init__(self, channels_img, features_d, use_batch_norm=True, activation_type="leaky_relu"):        super(Discriminator, self).__init__()        if activation_type == "relu":            act_fn = nn.ReLU(True)        elif activation_type == "leaky_relu":            act_fn = nn.LeakyReLU(0.2, inplace=True)        elif activation_type == "tanh":            act_fn = nn.Tanh()        else:            raise ValueError(f"Unsupported activation type: {activation_type}")        self.disc = nn.Sequential(            # Layer 1 (No BN on input like the paper)            nn.Conv2d(channels_img, features_d, kernel_size=4, stride=2, padding=1, bias=True),            act_fn,            # Layer 2            nn.Conv2d(features_d, features_d * 2, kernel_size=4, stride=2, padding=1, bias=not use_batch_norm),            *([nn.BatchNorm2d(features_d * 2)] if use_batch_norm else []),            act_fn,            # Layer 3            nn.Conv2d(features_d * 2, features_d * 4, kernel_size=4, stride=2, padding=1, bias=not use_batch_norm),            *([nn.BatchNorm2d(features_d * 4)] if use_batch_norm else []),            act_fn,            # Layer 4 (Output)            nn.Conv2d(features_d * 4, 1, kernel_size=4, stride=1, padding=0, bias=True), # No BN on output            nn.Sigmoid(),        )    def forward(self, x):        return self.disc(x)